In [4]:
# Librerías base
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pyreadstat
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.style.use('ggplot')

In [5]:
# Carga del diccionario CIE-10
cie10 = pd.read_excel(
    "defunciones/def_variables.xlsx",
    sheet_name="CIE-10",
    header=None,
    skiprows=2,        # saltamos fila 1 (título) y fila 2 (encabezado)
    usecols="A,B",
    names=["codigo", "descripcion"]
)

print(f"Total de códigos CIE-10 cargados: {len(cie10)}")
print(f"\nPrimeros registros:")
cie10.head(10)

Total de códigos CIE-10 cargados: 14361

Primeros registros:


,codigo,descripcion
0,A00,Cólera
1,A000,"Cólera debido a Vibrio cholerae 01, biotipo ch..."
2,A001,"Cólera debido a Vibrio cholerae 01, biotipo el..."
3,A009,"Cólera, no especificado"
4,A01,Fiebres tifoidea y paratifoidea
5,A010,Fiebre tifoidea
6,A011,Fiebre paratifoidea A
7,A012,Fiebre paratifoidea B
8,A013,Fiebre paratifoidea C
9,A014,"Fiebre paratifoidea, no especificada"


In [6]:
# Auditoría del diccionario CIE-10
print("=== NULOS ===")
print(cie10.isnull().sum())

print(f"\n=== DUPLICADOS EN CÓDIGO ===")
print(f"Códigos duplicados: {cie10['codigo'].duplicated().sum()}")

print(f"\n=== FORMATO DE CÓDIGOS ===")
print(f"Longitud mínima: {cie10['codigo'].str.len().min()}")
print(f"Longitud máxima: {cie10['codigo'].str.len().max()}")
print(f"¿Contiene puntos?: {cie10['codigo'].str.contains('.', regex=False).sum()} códigos")
print(f"¿Contiene espacios?: {cie10['codigo'].str.contains(' ').sum()} códigos")

print(f"\n=== MUESTRA POR LONGITUD ===")
for l in sorted(cie10['codigo'].str.len().unique()):
    ejemplo = cie10[cie10['codigo'].str.len() == l]['codigo'].iloc[0]
    count = (cie10['codigo'].str.len() == l).sum()
    print(f"  Longitud {l}: {count:,} códigos  →  ejemplo: '{ejemplo}'")

=== NULOS ===
codigo         0
descripcion    0
dtype: int64

=== DUPLICADOS EN CÓDIGO ===
Códigos duplicados: 1

=== FORMATO DE CÓDIGOS ===
Longitud mínima: 3
Longitud máxima: 4
¿Contiene puntos?: 0 códigos
¿Contiene espacios?: 0 códigos

=== MUESTRA POR LONGITUD ===
  Longitud 3: 1,800 códigos  →  ejemplo: 'A00'
  Longitud 4: 12,561 códigos  →  ejemplo: 'A000'


In [7]:
# Identificar y eliminar duplicado
duplicado = cie10[cie10['codigo'].duplicated(keep=False)]
print("=== CÓDIGO DUPLICADO ===")
print(duplicado)

# Limpieza: quedarse con la primera ocurrencia
cie10 = cie10.drop_duplicates(subset='codigo', keep='first').reset_index(drop=True)
print(f"\nDiccionario limpio: {len(cie10):,} códigos únicos")

=== CÓDIGO DUPLICADO ===
      codigo                                        descripcion
14332   U129  Vacunas covid-19 que causan efectos adversos e...
14333   U129  Vacunas covid-19 que causan efectos adversos e...

Diccionario limpio: 14,360 códigos únicos


In [8]:
# Inspección de un año de defunciones para entender la estructura
df_muestra, meta = pyreadstat.read_sav("defunciones/def2022.sav")

print(f"Registros: {len(df_muestra):,}")
print(f"Variables: {df_muestra.shape[1]}")
print(f"\n=== COLUMNAS Y TIPOS ===")
print(df_muestra.dtypes)
print(f"\n=== PRIMERAS FILAS ===")
df_muestra.head()

Registros: 95,386
Variables: 27

=== COLUMNAS Y TIPOS ===
Depreg     float64
Mupreg         str
Mesreg     float64
Añoreg     float64
Depocu     float64
Mupocu         str
Sexo       float64
Diaocu     float64
Mesocu     float64
Añoocu     float64
Edadif     float64
Perdif     float64
Puedif     float64
Ecidif     float64
Escodif    float64
Ciuodif        str
Pnadif     float64
Dnadif     float64
Mnadif         str
Nacdif     float64
Predif     float64
Dredif     float64
Mredif         str
Caudef         str
Asist      float64
Ocur       float64
Cerdef     float64
dtype: object

=== PRIMERAS FILAS ===


,Depreg,Mupreg,Mesreg,Añoreg,Depocu,Mupocu,Sexo,Diaocu,Mesocu,Añoocu,Edadif,Perdif,Puedif,Ecidif,Escodif,Ciuodif,Pnadif,Dnadif,Mnadif,Nacdif,Predif,Dredif,Mredif,Caudef,Asist,Ocur,Cerdef
0,13.0,1301,5.0,2022.0,13.0,1301,1.0,17.0,5.0,2022.0,39.0,3.0,5.0,1.0,2.0,92,484.0,23.0,2300,484.0,320.0,13.0,1315,X482,1.0,1.0,1.0
1,13.0,1318,9.0,2022.0,13.0,1318,2.0,17.0,8.0,2022.0,33.0,3.0,5.0,1.0,1.0,97,484.0,23.0,2300,484.0,320.0,13.0,1318,O720,5.0,6.0,9.0
2,13.0,1301,6.0,2022.0,13.0,1301,1.0,2.0,6.0,2022.0,39.0,3.0,5.0,2.0,2.0,92,484.0,23.0,2300,484.0,320.0,13.0,1326,X482,1.0,1.0,1.0
3,13.0,1326,4.0,2022.0,13.0,1326,2.0,22.0,4.0,2022.0,35.0,3.0,5.0,1.0,1.0,97,484.0,23.0,2300,484.0,320.0,13.0,1326,W749,4.0,9.0,9.0
4,14.0,1420,2.0,2022.0,14.0,1420,1.0,17.0,1.0,2022.0,35.0,3.0,5.0,1.0,1.0,97,484.0,23.0,2300,484.0,320.0,14.0,1420,W849,9.0,9.0,9.0


In [9]:
# Auditoría de la columna Caudef
caudef = df_muestra['Caudef'].copy()

print(f"=== NULOS ===")
print(f"Nulos en Caudef: {caudef.isnull().sum():,} ({caudef.isnull().mean()*100:.1f}%)")

print(f"\n=== FORMATO ===")
caudef_notnull = caudef.dropna()
print(f"Longitud mínima: {caudef_notnull.str.len().min()}")
print(f"Longitud máxima: {caudef_notnull.str.len().max()}")
print(f"¿Contiene puntos?: {caudef_notnull.str.contains('.', regex=False).sum()} registros")
print(f"¿Contiene espacios?: {caudef_notnull.str.contains(' ').sum()} registros")
print(f"¿Contiene minúsculas?: {caudef_notnull.str.contains('[a-z]').sum()} registros")

print(f"\n=== MATCH CON DICCIONARIO CIE-10 ===")
codigos_cie = set(cie10['codigo'])
match = caudef_notnull.isin(codigos_cie).sum()
no_match = (~caudef_notnull.isin(codigos_cie)).sum()
print(f"Con match:    {match:,} ({match/len(caudef_notnull)*100:.1f}%)")
print(f"Sin match:    {no_match:,} ({no_match/len(caudef_notnull)*100:.1f}%)")

print(f"\n=== EJEMPLOS SIN MATCH ===")
print(caudef_notnull[~caudef_notnull.isin(codigos_cie)].value_counts().head(15))

=== NULOS ===
Nulos en Caudef: 0 (0.0%)

=== FORMATO ===
Longitud mínima: 4
Longitud máxima: 4
¿Contiene puntos?: 0 registros
¿Contiene espacios?: 0 registros
¿Contiene minúsculas?: 0 registros

=== MATCH CON DICCIONARIO CIE-10 ===
Con match:    95,386 (100.0%)
Sin match:    0 (0.0%)

=== EJEMPLOS SIN MATCH ===
Series([], Name: count, dtype: int64)


In [10]:
# Auditoría de variables demográficas clave
cols_clave = ['Depocu', 'Mupocu', 'Sexo', 'Diaocu', 'Mesocu', 'Añoocu', 'Edadif', 'Perdif']

print("=== NULOS POR VARIABLE ===")
print(df_muestra[cols_clave].isnull().sum().to_string())

print("\n=== ESTADÍSTICAS DESCRIPTIVAS ===")
print(df_muestra[cols_clave].describe().round(2).to_string())

print("\n=== VALORES ÚNICOS - VARIABLES CATEGÓRICAS ===")
for col in ['Sexo', 'Depocu', 'Perdif']:
    print(f"\n{col}: {sorted(df_muestra[col].dropna().unique().tolist())}")

=== NULOS POR VARIABLE ===
Depocu    0
Mupocu    0
Sexo      0
Diaocu    0
Mesocu    0
Añoocu    0
Edadif    0
Perdif    0

=== ESTADÍSTICAS DESCRIPTIVAS ===
         Depocu      Sexo    Diaocu    Mesocu   Añoocu    Edadif    Perdif
count  95386.00  95386.00  95386.00  95386.00  95386.0  95386.00  95386.00
mean       8.72      1.45     15.67      6.38   2022.0     60.44      2.91
std        6.65      0.50      8.77      3.54      0.0     54.25      0.51
min        1.00      1.00      1.00      1.00   2022.0      0.00      1.00
25%        1.00      1.00      8.00      3.00   2022.0     40.00      3.00
50%        9.00      1.00     16.00      6.00   2022.0     64.00      3.00
75%       14.00      2.00     23.00      9.00   2022.0     79.00      3.00
max       22.00      2.00     31.00     12.00   2022.0    999.00      9.00

=== VALORES ÚNICOS - VARIABLES CATEGÓRICAS ===

Sexo: [1.0, 2.0]

Depocu: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0

In [11]:
# Leer solo columna A de la hoja Defunciones para ver qué variables existen
hoja_def = pd.read_excel(
    "defunciones/def_variables.xlsx",
    sheet_name="Defunciones",
    header=None,
    usecols="A",
    names=["variable"]
)

# Filtrar solo filas que tienen nombre de variable (no vacías)
variables_doc = hoja_def['variable'].dropna().reset_index(drop=True)
print(f"Variables documentadas en el diccionario: {len(variables_doc)}")
print()
for i, v in enumerate(variables_doc):
    print(f"  {i+1:02d}. {v}")

Variables documentadas en el diccionario: 29

  01. Valores de las variables defunciones
  02. Valor
  03. Departamento de registro
  04. Municipio de registro
  05. Mes de registro
  06. Año de registro
  07. Departamento de ocurrencia
  08. Municipio de ocurrencia
  09. Sexo del difunto(a)
  10. Día de ocurrencia
  11. Mes de ocurrencia
  12. Año ocurrencia
  13. Edad del difunto(a)
  14. Periodo de edad del difunto(a)
  15. Pueblo de pertenencia del difunto(a)
  16. Estado civil del difunto(a)
  17. Escolaridad del difunto(a)
  18. Ocupación (Subgrupos CIUO-08) del difunto(a)
  19. Pais de nacimiento del difunto(a)
  20. Departamento de nacimiento del difunto(a)
  21. Municipio de nacimiento del difunto(a)
  22. Nacionalidad del difunto(a)
  23. Pais de residencia del difunto(a)
  24. Departamento de residencia del difunto(a)
  25. Municipio de residencia del difunto(a)
  26. Causa de defuncion
  27. Asistencia recibida
  28. Sitio de ocurrencia
  29. Quien certifica


In [12]:
# Carga de todos los años de defunciones (solo columnas necesarias)
COLS_DEF = ['Depocu', 'Mupocu', 'Sexo', 'Diaocu', 'Mesocu', 'Añoocu', 'Edadif', 'Perdif', 'Caudef']
AÑOS_DEF  = range(2011, 2023)

frames = []
for año in AÑOS_DEF:
    df_año, _ = pyreadstat.read_sav(
        f"defunciones/def{año}.sav",
        usecols=COLS_DEF
    )
    frames.append(df_año)
    print(f"  def{año}.sav → {len(df_año):,} registros")

defunciones = pd.concat(frames, ignore_index=True)
print(f"\nTotal consolidado: {len(defunciones):,} registros | {defunciones.shape[1]} variables")

  def2011.sav → 72,354 registros
  def2012.sav → 72,657 registros
  def2013.sav → 76,639 registros
  def2014.sav → 77,807 registros
  def2015.sav → 80,876 registros
  def2016.sav → 82,565 registros
  def2017.sav → 81,726 registros
  def2018.sav → 83,071 registros
  def2019.sav → 85,600 registros
  def2020.sav → 96,001 registros
  def2021.sav → 118,465 registros
  def2022.sav → 95,386 registros

Total consolidado: 1,023,147 registros | 9 variables


In [13]:
# Auditoría del dataset consolidado
print("=== NULOS POR VARIABLE ===")
nulos = defunciones.isnull().sum()
pct   = (nulos / len(defunciones) * 100).round(2)
print(pd.DataFrame({'Nulos': nulos, 'Porcentaje': pct}).to_string())

print(f"\n=== VALORES ESPECIALES ===")
print(f"Edadif = 999:  {(defunciones['Edadif'] == 999).sum():,}")
print(f"Sexo   = 9:    {(defunciones['Sexo']   == 9.0).sum():,}")
print(f"Depocu = 99:   {(defunciones['Depocu'] == 99).sum():,}")
print(f"Perdif únicos: {sorted(defunciones['Perdif'].dropna().unique().tolist())}")

print(f"\n=== RANGO DE AÑOS ===")
print(defunciones['Añoocu'].value_counts().sort_index().to_string())

=== NULOS POR VARIABLE ===
         Nulos  Porcentaje
Depocu       0        0.00
Sexo         0        0.00
Diaocu       0        0.00
Mesocu       0        0.00
Edadif       0        0.00
Perdif       0        0.00
Caudef       0        0.00
Mupocu   72354        7.07
Añoocu  299457       29.27

=== VALORES ESPECIALES ===
Edadif = 999:  7,022
Sexo   = 9:    0
Depocu = 99:   0
Perdif únicos: [1.0, 2.0, 3.0, 9.0]

=== RANGO DE AÑOS ===
Añoocu
2015.0     80876
2016.0     82565
2017.0     81726
2018.0     83071
2019.0     85600
2020.0     96001
2021.0    118465
2022.0     95386


In [14]:
# Inspeccionar columnas reales de los 4 años problemáticos
años_problematicos = [2011, 2012, 2013, 2014]

for año in años_problematicos:
    df_temp, meta = pyreadstat.read_sav(f"defunciones/def{año}.sav")
    print(f"\ndef{año}.sav — {df_temp.shape[1]} columnas:")
    print(list(df_temp.columns))


def2011.sav — 26 columnas:
['Depreg', 'mupreg', 'Mesreg', 'Añoreg', 'Depocu', 'mupocu', 'Areag', 'Sexo', 'Diaocu', 'Mesocu', 'añoocu', 'Edadif', 'Perdif', 'Getdif', 'Ecidif', 'Escodif', 'Ocudif', 'Dnadif', 'Mnadif', 'Nacdif', 'Dredif', 'Mredif', 'Caudef', 'Asist', 'Ocur', 'Cerdef']

def2012.sav — 27 columnas:
['Depreg', 'Mupreg', 'Mesreg', 'Añoreg', 'Depocu', 'Mupocu', 'Areag', 'Sexo', 'Diaocu', 'Mesocu', 'Edadif', 'Perdif', 'Getdif', 'Ecidif', 'Escodif', 'Ocudif', 'Pnadif', 'Dnadif', 'Mnadif', 'Nacdif', 'Predif', 'Dredif', 'Mredif', 'Caudef', 'Asist', 'Ocur', 'Cerdef']

def2013.sav — 28 columnas:
['Depreg', 'Mupreg', 'Mesreg', 'Añoreg', 'Depocu', 'Mupocu', 'Areag', 'Sexo', 'Diaocu', 'Mesocu', 'Edadif', 'Perdif', 'Puedif', 'Ecidif', 'Escodif', 'Ciuodif', 'Pnadif', 'Dnadif', 'Mnadif', 'Nacdif', 'Predif', 'Dredif', 'Mredif', 'Caudef', 'caudef.descrip', 'Asist', 'Ocur', 'Cerdef']

def2014.sav — 27 columnas:
['Depreg', 'Mupreg', 'Mesreg', 'Añoreg', 'Depocu', 'Mupocu', 'Areag', 'Sexo', 'Di

In [15]:
# Recarga robusta manejando inconsistencias de naming entre años
COLS_TARGET = {
    'depocu'  : 'depocu',
    'mupocu'  : 'mupocu',
    'sexo'    : 'sexo',
    'diaocu'  : 'diaocu',
    'mesocu'  : 'mesocu',
    'edadif'  : 'edadif',
    'perdif'  : 'perdif',
    'caudef'  : 'caudef',
    'añoocu'  : 'anio',   # 2011 lo tiene con este nombre
}

frames = []
for año in range(2011, 2023):
    df_temp, _ = pyreadstat.read_sav(f"defunciones/def{año}.sav")

    # Normalizar todos los nombres a minúscula
    df_temp.columns = df_temp.columns.str.lower()

    # Renombrar al esquema estándar
    df_temp = df_temp.rename(columns=COLS_TARGET)

    # Si el año no vino en el archivo, inyectarlo
    if 'anio' not in df_temp.columns:
        df_temp['anio'] = año

    # Seleccionar solo columnas que necesitamos
    cols_finales = ['anio', 'depocu', 'mupocu', 'sexo', 'diaocu', 'mesocu', 'edadif', 'perdif', 'caudef']
    df_temp = df_temp[cols_finales]

    frames.append(df_temp)
    print(f"  def{año}.sav → {len(df_temp):,} registros | anio: {df_temp['anio'].iloc[0]:.0f}")

defunciones = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(defunciones):,} registros | {defunciones.shape[1]} variables")
print(f"Nulos en anio: {defunciones['anio'].isnull().sum()}")
print(f"\nDistribución por año:")
print(defunciones['anio'].value_counts().sort_index().to_string())

  def2011.sav → 72,354 registros | anio: 2011
  def2012.sav → 72,657 registros | anio: 2012
  def2013.sav → 76,639 registros | anio: 2013
  def2014.sav → 77,807 registros | anio: 2014
  def2015.sav → 80,876 registros | anio: 2015
  def2016.sav → 82,565 registros | anio: 2016
  def2017.sav → 81,726 registros | anio: 2017
  def2018.sav → 83,071 registros | anio: 2018
  def2019.sav → 85,600 registros | anio: 2019
  def2020.sav → 96,001 registros | anio: 2020
  def2021.sav → 118,465 registros | anio: 2021
  def2022.sav → 95,386 registros | anio: 2022

Total: 1,023,147 registros | 9 variables
Nulos en anio: 0

Distribución por año:
anio
2011.0     72354
2012.0     72657
2013.0     76639
2014.0     77807
2015.0     80876
2016.0     82565
2017.0     81726
2018.0     83071
2019.0     85600
2020.0     96001
2021.0    118465
2022.0     95386


In [19]:
# Inspección de columnas reales por año
AÑOS_NEC = [2011, 2012, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

for año in AÑOS_NEC:
    df_temp, _ = pyreadstat.read_sav(f"necropsias/nec{año}.sav")
    df_temp.columns = df_temp.columns.str.lower()
    print(f"\nnec{año}.sav — {df_temp.shape[1]} columnas:")
    print(list(df_temp.columns))


nec2011.sav — 7 columnas:
['num_corre', 'mes_ocu', 'depto_ocu', 'edad_person', 'g_edad', 'sexo_person', 'causa_muerte']

nec2012.sav — 7 columnas:
['num_corre', 'mes_ocu', 'depto_ocu', 'edad_person', 'g_edad', 'sexo_person', 'causa_muerte']

nec2016.sav — 13 columnas:
['no_corrre', 'año_ocu', 'mes_ocu', 'diasem_ocu', 'dia_ocu', 'depto_ocu', 'edad_per', 'sexo_per', 'causa_muerte', 'g_edad_80ymás', 'g_edad_60ymás', 'edad_quinquenales', 'mayor_menor']

nec2017.sav — 14 columnas:
['núm_corre', 'año_ocu', 'mes_ocu', 'día_ocu', 'día_sem_ocu', 'depto_ocu', 'mupio_ocu', 'sexo_per_eva', 'edad_per', 'g_edad_60ymás', 'g_edad_80ymás', 'edad_quinquenales', 'menor_mayor', 'eva_mn']

nec2018.sav — 14 columnas:
['núm_corre', 'año_ocu', 'mes_ocu', 'día_ocu', 'dia_sem_ocu', 'depto_ocu', 'mupio_ocu', 'edad_per', 'g_edad_60ymás', 'g_edad_80ymás', 'edad_quinquenales', 'menor_mayor', 'sexo_per_eva', 'eva_mn']

nec2019.sav — 14 columnas:
['núm_corre', 'año_ocu', 'mes_ocu', 'día_ocu', 'día_sem_ocu', 'depto_o

In [20]:
for año in [2017, 2018]:
    df_temp, _ = pyreadstat.read_sav(f"necropsias/nec{año}.sav")
    df_temp.columns = df_temp.columns.str.lower()
    print(f"\nnec{año} — eva_mn únicos ({df_temp['eva_mn'].nunique()}):")
    print(df_temp['eva_mn'].value_counts().head(20))


nec2017 — eva_mn únicos (42):
eva_mn
20.0    3885
37.0    3489
2.0     1317
21.0     488
19.0     473
27.0     357
22.0     313
23.0     301
7.0      171
18.0     160
29.0     147
33.0     132
5.0       89
32.0      81
16.0      70
38.0      58
8.0       39
3.0       32
11.0      29
31.0      28
Name: count, dtype: int64

nec2018 — eva_mn únicos (39):
eva_mn
20.0    3491
37.0    3359
2.0     1347
21.0     560
27.0     416
19.0     408
22.0     391
23.0     248
18.0     229
29.0     165
7.0      158
33.0     119
5.0      102
16.0      77
38.0      77
32.0      53
8.0       47
31.0      31
4.0       24
11.0      21
Name: count, dtype: int64


In [21]:
hoja_nec = pd.read_excel(
    "necropsias/nec2018.xlsx",
    header=None,
    usecols="A",
    names=["variable"]
)

variables_nec = hoja_nec['variable'].dropna().reset_index(drop=True)
print(f"Variables documentadas: {len(variables_nec)}")
for i, v in enumerate(variables_nec):
    print(f"  {i+1:02d}. {v}")

Variables documentadas: 17
  01. NECROPSIAS
  02. Valores de las variables
  03. Valor
  04. NÚMERO DE CORRELATIVO
  05. AÑO DE OCURRENCIA
  06. MES DE OCURRENCIA
  07. DIA DE OCURRENCIA
  08. DIA DE LA SEMANA DE OCURRENCIA
  09. DEPARTAMENTO DE OCURRENCIA
  10. MUNICIPIO DE OCURRENCIA
  11. SEXO DE LA PERSONA EVALUADA
  12. EDAD SIMPLE
  13. GRUPO DE EDAD QUINQUENAL (MENOR 15 HASTA 60 Y MÁS)
  14. GRUPO DE EDAD QUINQUENAL (MENOR 15 HASTA 80 Y MÁS)
  15. EDADES QUINQUENALES (A PARTIR DE 0 HASTA 80 Y MÁS)
  16. MENOR O MAYOR DE EDAD
  17. EVALUACIÓN MÉDICO DE NECROPSIA


In [22]:
# Leer diccionario de Evaluación Médico de Necropsia
dic_eva_mn = pd.read_excel(
    "necropsias/nec2018.xlsx",
    header=None,
    skiprows=446,        # empieza en fila 447 (índice 446)
    usecols="B,C",
    names=["codigo", "descripcion"]
).dropna().reset_index(drop=True)

print(f"Total de códigos eva_mn: {len(dic_eva_mn)}")
print(dic_eva_mn.to_string())

Total de códigos eva_mn: 42
    codigo                                                                                descripcion
0        1                                                                                 Amputación
1        2                                                                                    Asfixia
2        3                                                                             Cardiomiopatía
3        4                                                                               Decapitación
4        5                                                                                  Depleción
5        6                                                                               Desnutrición
6        7                                                                             Edema pulmonar
7        8                                                                              Electrocución
8        9                                            

In [25]:
print(dic_eva_mn[dic_eva_mn['codigo'].between(24, 39)].to_string())

    codigo                         descripcion
23      24           Malformaciones congénitas
24      25                          Meningitis
25      26  Mordedura y/o picadura de insectos
26      27                            Neumonía
27      28    Otras enfermedades al pericardio
28      29                        Pancreatitis
29      30                         Peritonitis
30      31                          Prematurez
31      32                           Quemadura
32      33                              Sepsis
33      35               Taponamiento cardiaco
34      36             Transtornos metabólicos
35      37                         Traumatismo
36      38                       Tromboembolia
37      39                           Trombosis
